# Complete the ERA5 initial states with the missing model fields

## Purpose

The ERA5 initial files `v3.LR.ne30pg2.ERA5.eam.i.<date>.nc`, used by HYB_BASE (through `add_dart_perturbations_to_hyb_clim.ipynb`) and DA_LND, hold only T, U, V, Q, PS, CLDLIQ and CLDICE. A full E3SMv3 initial file also holds ozone, the Linoz tracers, chemistry, MAM aerosols, hydrometeor numbers and surface fields, and EAM starts any missing field from zero. The radiation reads its gases from tracers (`rad_climate = 'A:H2OLNZ:H2O', 'A:O3:O3', 'A:N2OLNZ:N2O', 'A:CH4LNZ:CH4'`), so those runs started radiatively dry and without ozone: on day 1 about 190 W m-2 less downward longwave at the surface and 250 DU less ozone than DA_ATM, and a 6-7 K cold 2-m temperature over land into week 3 (`diagnostic/jupyter/hind_lead_eval`).

## Method

For each start date, `util/complete_era5_ic.py`:

1. averages every state field of the season's 20 selected DART members (the members the HYB perturbations and the DA runs use) into a file with the native layout (dynamics fields on `ncol_d`, physics-grid surface fields on `ncol`);
2. replaces T, U, V, Q, PS, CLDLIQ and CLDICE with the ERA5 values (same GLL columns and hybrid levels, checked);
3. sets the radiative water vapour to the ERA5 moisture with the model's own relative difference from `Q`: `H2OLNZ = Q_ERA5 * (mean H2OLNZ_DART / mean Q_DART)`, the ratio bounded to `H2O_RATIO_BOUNDS` (multiplicative, so it stays positive where ERA5 is drier than the DART mean; an additive offset went negative at 0.25 % of the points).

Checks per file: the ERA5 fields are exact, no DART field is missing, no field is non-finite, the ozone column is realistic, and `H2OLNZ/Q` is close to 1 in the lower troposphere.

The ensemble mean of the hydrometeor numbers goes with the ERA5 cloud water and ice; the microphysics adjusts them in the first steps.

## Outputs

`OUTPUT_DIR/v3.LR.ne30pg2.ERA5c.eam.i.<date>.nc` and `OUTPUT_DIR/completion_checks.csv`. The HYB perturbations are then rebuilt from these files (set the reference directory and file name in `add_dart_perturbations_to_hyb_clim.ipynb`, and add `H2OLNZ` to the perturbed variables), and DA_LND points its `ncdata` at them.

## 1. Libraries

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

## 2. User setup

Paths and the start dates with their DART members. Plain values only, so `papermill -p` overrides work.

In [2]:
ERA5_ATM_DIR = Path("/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM")           # v3.LR.ne30pg2.ERA5.eam.i.<date>.nc
DART_ROOT = Path("/global/cfs/cdirs/m4849/zhan391/e3sm_dart")                  # <case>/<member>/archive/rest/<date>/
OUTPUT_DIR = Path("/global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE")    # completed files
WORK_DIR = Path("/global/cfs/cdirs/m4849/zhan391/work_package/e3sm_dart_analysis")

# start dates -> DART case and the 20 selected DART members of that season
DATE_SETS = {
    "dec2011": dict(dates=["2011-12-16-00000", "2011-12-21-00000", "2011-12-26-00000", "2011-12-31-00000"],
                    case="DARTEN40_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
                    members=[1, 3, 4, 7, 8, 9, 10, 14, 17, 18, 20, 21, 25, 26, 28, 31, 35, 36, 37, 40]),
    "may2012": dict(dates=["2012-05-16-00000", "2012-05-21-00000", "2012-05-26-00000", "2012-05-31-00000"],
                    case="DARTEN40S1_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy",
                    members=[2, 3, 5, 7, 11, 12, 13, 14, 15, 16, 20, 22, 24, 26, 27, 28, 31, 32, 34, 38]),
}
OUTPUT_PATTERN = "v3.LR.ne30pg2.ERA5c.eam.i.{date}.nc"
H2O_RATIO_BOUNDS = (0.2, 5.0)
OVERWRITE = False

## 3. Paths and modules

In [3]:
pkg = WORK_DIR.resolve() / "ensemble_perturbations"
if str(pkg) not in sys.path:
    sys.path.insert(0, str(pkg))
from util.complete_era5_ic import complete_era5_ic, dart_member_file, validate

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
jobs = []
for name, ds in DATE_SETS.items():
    files_of = lambda d: [dart_member_file(DART_ROOT, ds["case"], f"EN{k:02d}", d) for k in ds["members"]]
    for d in ds["dates"]:
        jobs.append(dict(set=name, date=d, era5=ERA5_ATM_DIR / f"v3.LR.ne30pg2.ERA5.eam.i.{d}.nc",
                         dart=files_of(d), out=OUTPUT_DIR / OUTPUT_PATTERN.format(date=d)))
missing = [str(p) for j in jobs for p in [j["era5"], *j["dart"]] if not p.exists()]
if missing:
    raise FileNotFoundError(f"{len(missing)} inputs missing, e.g. {missing[:3]}")
print(f"[INPUT] {len(jobs)} start dates, {sum(len(j['dart']) for j in jobs)} DART member files\n[OUTPUT] {OUTPUT_DIR}")

[INPUT] 8 start dates, 160 DART member files
[OUTPUT] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE


## 4. Complete and check

Existing outputs are only re-checked unless `OVERWRITE`.

In [4]:
rows = []
for j in jobs:
    if j["out"].exists() and not OVERWRITE:
        r = validate(j["out"], j["era5"], j["dart"][0])
    else:
        r = complete_era5_ic(j["era5"], j["dart"], j["out"], ratio_bounds=H2O_RATIO_BOUNDS, overwrite=OVERWRITE)
    rows.append(dict(set=j["set"], date=j["date"], **{k: v for k, v in r.items() if k != "file"}))
    print(f"[WRITE] {j['out']}")
checks = pd.DataFrame(rows)
checks.to_csv(OUTPUT_DIR / "completion_checks.csv", index=False)
display(checks)

Warning 3: Cannot find header.dxf (GDAL_DATA is not defined)
ERROR 1: PROJ: proj_create_from_database: Cannot find proj.db


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2011-12-16-00000.nc


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2011-12-21-00000.nc


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2011-12-26-00000.nc


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2011-12-31-00000.nc


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2012-05-16-00000.nc


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2012-05-21-00000.nc


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2012-05-26-00000.nc


[WRITE] /global/cfs/cdirs/m4849/zhan391/DART_INIT/ATM_COMPLETE/v3.LR.ne30pg2.ERA5c.eam.i.2012-05-31-00000.nc


,set,date,missing_vs_dart,era5_max_abs_diff,nonfinite_vars,toz_du_mean,toz_du_min,toz_du_max,h2olnz_over_q_below_500hPa_median,n_vars,h2olnz_ratio_bounded_points
0,dec2011,2011-12-16-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],275.974192,211.973218,450.927898,0.992023,129,5629
1,dec2011,2011-12-21-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],276.672246,204.087491,469.983390,0.991016,129,8007
2,dec2011,2011-12-26-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],277.759683,195.503498,469.360743,0.991622,129,7194
3,dec2011,2011-12-31-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],279.246041,212.452840,496.615387,0.991291,129,9414
4,may2012,2012-05-16-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],279.856654,224.083128,387.527491,0.991424,129,5212
5,may2012,2012-05-21-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],280.646089,224.742871,391.030851,0.992025,129,6790
6,may2012,2012-05-26-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],281.391726,219.154324,413.956053,0.990804,129,8570
7,may2012,2012-05-31-00000,[],"{'T': 0.0, 'U': 0.0, 'V': 0.0, 'Q': 0.0, 'PS':...",[],282.236969,218.179744,443.745875,0.991512,129,12600


## 5. Verdict

In [5]:
bad = checks[(checks.missing_vs_dart.str.len() > 0) | (checks.nonfinite_vars.str.len() > 0)
             | (checks.era5_max_abs_diff.apply(lambda d: max(d.values())) > 0)
             | ~checks.toz_du_mean.between(200, 400)]
print("[CHECK] all files pass" if bad.empty else f"[CHECK] failing files:\n{bad[['set', 'date']]}")

[CHECK] all files pass
